# Udhëtimet dhe analiza shumëvariablore

Ky notebook prodhon përmbledhjet e udhëtimeve, krahason metodat e detektimit dhe përgatit paketën e të dhënave. Tabelat dhe figurat ruhen nga funksionet e pipeline-it.

Përpara ekzekutimit, përgatit tabelat hyrëse me komandën `python -m taxi.pipeline run --dev` për zhvillim ose `python -m taxi.pipeline run` për të dhënat e plota.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from taxi import config, pipeline

DEV = False
config.set_dev(DEV)


## Klasat dhe rregullat e udhëtimeve

`stationary` tregon episode me taksimetër aktiv dhe distancë nën pragun e konfiguruar. Udhëtimet e tjera janë `ride`. Rregullat e shpejtësisë, distancës dhe kohëzgjatjes përshkruhen veçmas dhe nuk i fshijnë udhëtimet.


In [ ]:
pipeline.run(stage="explore-trips", dev=DEV)
trip_rules = pd.read_csv(config.tables_dir() / "outliers_trips.csv")
trip_summary = pd.read_csv(config.tables_dir() / "summary_trips.csv")
display(trip_rules.head(20))
display(trip_summary.head(20))


## Detektimi shumëvariablor

Krahasohen distanca Mahalanobis robuste, Local Outlier Factor, Isolation Forest dhe gabimi i rindërtimit nga PCA. Përqindja e anomalive nuk fiksohet paraprakisht. Rastet kryesore që presin shqyrtim shënohen `unreviewed`; ato duhen lidhur me pikët përkatëse dhe klasifikuar pas kontrollit.


In [ ]:
multivariate = pd.read_csv(config.tables_dir() / "outliers_multivariate.csv")
method_counts = multivariate[multivariate["row_type"] == "method"]
review_queue = multivariate[multivariate["row_type"].isna() | multivariate["index"].notna()]
display(method_counts)
display(review_queue.head(40))


## Përmbledhja shumëvariablore

Matrica përmban kovariancat dhe korrelacionet Pearson e Spearman. Testet e tabelave të kryqëzuara shoqërohen me madhësinë e efektit Cramér's V; vlerat p interpretohen bashkë me madhësinë e efektit dhe numrin e madh të udhëtimeve.


In [ ]:
multivariate_summary = pd.read_csv(config.tables_dir() / "multivariate_summary.csv")
od_matrix = pd.read_csv(config.tables_dir() / "od_matrix.csv")
display(multivariate_summary[multivariate_summary["section"] == "contingency_test"])
display(multivariate_summary[multivariate_summary["section"].isin(["pca_after_screening", "pca_loading_change", "pca_variance_change"])].head(40))
display(od_matrix.head(30))


In [ ]:
for name in ["f19_trip_classes.png", "f20_multivariate_outliers.png", "f21_correlation_od_matrix.png"]:
    path = config.figures_dir() / name
    if path.exists():
        display(Image(filename=str(path)))


## Eksporti

Paketa përfshin pikët me flamujt e tyre, udhëtimet, tabelat model, tabelat e analizës dhe fjalorin e të dhënave. Kontrolli i eksportit ndalet nëse një kolonë e dataset-eve të eksportuara mungon nga fjalori.


In [ ]:
pipeline.run(stage="export", dev=DEV)
manifest = pd.read_csv(config.tables_dir() / "export_manifest.csv")
display(manifest.head(20))
print(f"Madhësia e paketës: {manifest['bytes'].sum():,} bytes, pa README_data.md")
